##### Q1: Build Your Personalized Knowledge Base

Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits as follows:

- Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account", "general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g. if d%3 gives "account", write a question like “how do I update my registered mobile number”).
- ##### Example roll number ...23 -> digits 2, 3
- ##### digit 2 -> category[2 % 3] = general
- ##### digit 3 -> category[3 % 3] = billing

**Output:** Print your final 6-row DataFrame.

In [2]:
import pandas as pd
roll_number = 1024170405
last_two_digits = [int(d) for d in str(roll_number)[-2:]]
categories = ["billing", "account", "general"]
fixed_entries = [
    {"question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"},
    {"question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"},
    {"question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"},
    {"question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"}
]

for digit in last_two_digits:
    category = categories[digit % 3]

    if category == "billing":
        question = "how can i check my transaction history"
        answer = "You can check your transaction history from the Billing section."
        keywords = "transaction history billing payment"

    elif category == "account":
        question = "how do i update my registered mobile number"
        answer = "Go to Account Settings and update your registered mobile number."
        keywords = "mobile number update account"

    else:
        question = "how can i contact customer support"
        answer = "You can contact customer support during working hours."
        keywords = "support help contact"

    fixed_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })
df = pd.DataFrame(fixed_entries)
print(df)

                                 question  \
0                  what is the annual fee   
1                   how to reset password   
2             what are your working hours   
3                   how can i pay the fee   
4  how can i check my transaction history   
5      how can i contact customer support   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can check your transaction history from th...   
5  You can contact customer support during workin...   

                              keywords category  
0                fee cost price charge  billing  
1                 password reset login  account  
2               hours timing open time  general  
3                  pay payment upi fee  billing  
4  transaction history billing payment  b

#### Q2: Generate and Score a Hypothesis :Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.

In [3]:
def score_query(query, df):
    query_terms = set(query.lower().split())
    matches = []

    for _, faq in df.iterrows():
        question_words = set(faq["question"].lower().split())
        keyword_words = set(faq["keywords"].lower().split())

        keyword_matches = query_terms.intersection(keyword_words)
        question_matches = query_terms.intersection(question_words)

        confidence = 2 * len(keyword_matches) + len(question_matches)

        if confidence > 0:
            matches.append({
                "question": faq["question"],
                "answer": faq["answer"],
                "category": faq["category"],
                "confidence": confidence
            })

    matches.sort(key=lambda x: x["confidence"], reverse=True)
    return pd.DataFrame(matches)

query = "transaction history"
result = score_query(query, df)
print(result)

                                 question  \
0  how can i check my transaction history   

                                              answer category  confidence  
0  You can check your transaction history from th...  billing           6  


#### Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [5]:
def same_category(category_name, df):
    return df[df["category"] == category_name][["question", "answer", "keywords", "category"]]

personalized_category = df.iloc[4]["category"]
print("Category:", personalized_category)
print(same_category(personalized_category, df))

Category: billing
                                 question  \
0                  what is the annual fee   
3                   how can i pay the fee   
4  how can i check my transaction history   

                                              answer  \
0                          The annual fee is Rs 500.   
3         You can pay via UPI, card, or net banking.   
4  You can check your transaction history from th...   

                              keywords category  
0                fee cost price charge  billing  
3                  pay payment upi fee  billing  
4  transaction history billing payment  billing  


#### Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named your_roll_number_faq_data.csv.

In [ ]:
entry_index = 4
new_keyword = input("Enter a new keyword: ")

df.loc[entry_index, "keywords"] += " " + new_keyword

filename = "1024170405_faq_data.csv"
df.to_csv(filename, index=False)

print(df.loc[entry_index])
print("Saved as:", filename)

### Q5: Using groupby, print how many FAQ entries you have per category.

In [36]:
category_counts = df.groupby("category").size()
print(category_counts)

category
account    2
billing    2
general    2
dtype: int64


### Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [41]:
def score_query_with_ties(query, df):
    results = []

    for index, row in df.iterrows():
        score = 0

        if query.lower() in row["question"].lower():
            score += 2

        if query.lower() in row["keywords"].lower():
            score += 1

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    if len(results) == 0:
        print("No matching entries found.")
        return

    result_df = pd.DataFrame(results)

    highest_score = result_df["confidence"].max()
    best_matches = result_df[result_df["confidence"] == highest_score]

    if len(best_matches) > 1:
        print("Tie detected! Multiple entries have the highest score:")
    else:
        print("Single best match found:")

    print(best_matches)
score_query_with_ties("fee", df)
score_query_with_ties("password", df)

Tie detected! Multiple entries have the highest score:
                 question                                      answer  \
0  what is the annual fee                   The annual fee is Rs 500.   
1   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  confidence  
0  billing           3  
1  billing           3  
Single best match found:
                question                            answer category  \
0  how to reset password  Go to Settings > Reset Password.  account   

   confidence  
0           3  
